## 1 - Instalando as bibliotecas

**setence-transformers** - Embbedings (Biblioteca que converte text, audio, video em vetores númericos)

**faiss-cpu** - Similiaridade de cosseno (Distância euclidiana)

**transformes** - Modelos de IA generativas

****


In [1]:
!pip uninstall -y sentence-transformers transformers huggingface-hub tokenizers accelerate

Found existing installation: sentence-transformers 5.5.1
Uninstalling sentence-transformers-5.5.1:
  Successfully uninstalled sentence-transformers-5.5.1
Found existing installation: transformers 5.9.0
Uninstalling transformers-5.9.0:
  Successfully uninstalled transformers-5.9.0
Found existing installation: huggingface_hub 1.16.1
Uninstalling huggingface_hub-1.16.1:
  Successfully uninstalled huggingface_hub-1.16.1
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2


In [2]:
%pip install -q sentence-transformers faiss-cpu transformers pathlib pandas

Note: you may need to restart the kernel to use updated packages.


## 2 - Testando os imports e conectando meu google drive

Teste pra ver se tem GPU cuda para melhor treinamento e execução do modelo

In [3]:
import pandas as pd
import os
import numpy as np
import faiss
import torch
import textwrap
from pathlib import Path
import json
import warnings
import regex as re
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"dispositivo detectado: {device.upper()}")

if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   GPU: {gpu_name}")
    print(f"   VRAM disponível: {gpu_mem:.1f} GB")

dispositivo detectado: CUDA
   GPU: NVIDIA GeForce RTX 3060
   VRAM disponível: 12.9 GB


## 3 - Criando o DataFrame com os chamados da base de dados

Criando o df com base nos seguintes atributos:

### Campos do Dataset
| Campo | Descrição |
|---|---|
| `Id` | Identificador do chamado |
| `Título` | Titulo do chamado |
| `Descrição do chamado` | Descrição do chamado |
| `Última ação de acompanhamento` | Solução aplicada para o chamado |
| `Time` | Time que atendeu o chamado |
| `Tempo em horas do atendimento` | Tempo total em horas de atendimento |

In [4]:
# Nível 1 — Tópicos (saída do BERTopic + sumarização, granularidade baixa)
df_topicos = pd.read_csv('../data/chatbot/df_topicos.csv')

df_topicos = df_topicos.rename(columns={
    'Sistema':         'sistema_origem',
    'Tópico':          'topico_id_original',
    'Título':          'label_curto',
    'Nº Documentos':   'n_documentos',
})

df_topicos['topico_id_unico'] = (
    df_topicos['sistema_origem'].str.upper().str.replace(' ', '') +
    '_' +
    df_topicos['topico_id_original'].astype(str)
)

df_topicos['texto_para_embedding'] = (
    df_topicos['label_curto'] + '. ' +
    df_topicos['padrao_dominante'].fillna('') + ' ' +
    df_topicos['impacto_operacional'].fillna('')
).str.strip()


print(f"Topicos carregados: {len(df_topicos)}")
print(f"Colunas finais: {df_topicos.columns.tolist()}")
print(f"\nExemplo de texto_para_embedding:")
print(df_topicos['texto_para_embedding'].iloc[0])

# Nível 2 — Chamados (granularidade alta, já enriquecido com topico_id_unico)
df = pd.read_csv('../data/chatbot/df_chamados.csv')

print(f"Topicos carregados : {len(df_topicos)} (de {df_topicos['sistema_origem'].nunique()} sistemas)")
print(f"Chamados carregados: {len(df)}")

# Verificação de integridade adaptada para as VÍRGULAS
topicos_validos = set(df_topicos['topico_id_unico'])

def checar_se_tem_orfao(tags):
    if pd.isna(tags) or tags == "NAO_CLASSIFICADO":
        return True # É órfão (não tem tópico)
    lista_tags = str(tags).split(",")
    # Se TODOS os tópicos listados forem inválidos, é órfão
    return not any(t in topicos_validos for t in lista_tags)

chamados_orfaos = df[df['topico_id_unico'].apply(checar_se_tem_orfao)]

if len(chamados_orfaos) > 0:
    print(f"\nALERTA: {len(chamados_orfaos)} chamados com topico_id_unico não mapeado ou não classificado.")
else:
    print(f"\nIntegridade OK: todos os chamados mapeiam para um tópico válido!")

Topicos carregados: 34
Colunas finais: ['topico_id_unico', 'sistema_origem', 'topico_id_original', 'label_curto', 'resumo_sumarizado', 'n_documentos', 'keywords', 'padrao_dominante', 'impacto_operacional', 'texto_para_embedding']

Exemplo de texto_para_embedding:
Problemas de Afastamentos e Atestados no Sistema de RH. O padrão dominante nesse cluster de tickets é relacionado a problemas com o sistema de afastamentos e atestados, incluindo erros ao tentar lançar ou registrar afastamentos, inconsistências em registros de funcionários, problemas de integração entre sistemas e dificuldades para agendar perícias. Além disso, há também mencionadas falhas na validação de documentos e assinaturas digitais. Os usuários relatam que os erros persistem mesmo após tentativas de resolver o problema, o que causa atrasos nos processos e impacta negativamente no trabalho dos funcionários. As unidades precisam lançar afastamentos manualmente em duplicidade, gerando inconsistências nos registros de licen

## 4 - Junção de atributos

Juntar todos os atributos importantes citados acima para melhor contextualização sobre o problema para o modelo, além de facilitar o embbeding e a similiaridade de cosseno.

> **Teste futuro?**: A forma de combinar campos é um hiperparâmetro do sistema. Experimento sugerido: compare embeddings gerados apenas com `titulo` vs `titulo + descricao` vs texto completo.

In [5]:
def criar_texto_combinado(row):
    partes = [
        f"Título: {row['Título']}",
        f"Descrição: {row['Descrição do chamado']}"
    ]
    return " | ".join(partes)

df['texto_completo'] = df.apply(criar_texto_combinado, axis=1)

def limpar_texto(texto):
    texto = texto.strip()
    texto = ' '.join(texto.split())
    return texto

df['texto_completo'] = df['texto_completo'].apply(limpar_texto)

print(f"\nExemplo de texto combinado:")
print("─" * 60)
print(textwrap.fill(df['texto_completo'].iloc[0], width=70))
print("─" * 60)
print(f"\nComprimento médio dos textos: {df['texto_completo'].str.len().mean():.0f} caracteres")

def extrair_link_da_solucao(texto_solucao: str) -> str:
    if not isinstance(texto_solucao, str):
        return None

    # Regex padrão para identificar URLs (http, https, www)
    padrao_url = r'(https?://[^\s<>"]+|www\.[^\s<>"]+)'
    urls = re.findall(padrao_url, texto_solucao)

    return urls[0] if urls else None


Exemplo de texto combinado:
────────────────────────────────────────────────────────────
Título: acesso geral ao dw siapenet nao apenas ao meu proprio orgao |
Descrição: solicito novamente por gentileza acesso geral ao dw
siapenet que foi cancelado ha alguns meses ja precisamos desse acesso
geral para extrair os microdados e realizar as analises demanadadas
pelo mgi e outros orgao da administracao na solicitacao anterior o
acesso estava restrito a visualizacao dados do ipea cordialmente lopez
────────────────────────────────────────────────────────────

Comprimento médio dos textos: 635 caracteres


## 5 - Embbedings com o Setence-Bert

Vetorização dos documentos (chamados -> texto agrupado acima) de forma densa e com n-dimensões para analise de similaridade semântica.

### Modelo escolhido: `paraphrase-multilingual-MiniLM-L12-v2`

In [6]:
BASE_DIR = Path.cwd().parent 
METADATA_PATH = BASE_DIR / "chatbot" / "models" / "metadata.json"

caminho_modelo_local = None

if METADATA_PATH.exists():
    try:
        with open(METADATA_PATH, "r", encoding="utf-8") as f:
            metadata = json.load(f)
            caminho_modelo_local = metadata.get("embedding_model_local_path")
    except Exception as e:
        print(f"Aviso: Erro ao ler metadados: {e}")

# Valida se o diretório do modelo realmente existe
if caminho_modelo_local and os.path.exists(caminho_modelo_local):
    print(f"Carregando modelo de embeddings LOCAL do disco:\n{caminho_modelo_local}")
    MODELO_EMBEDDING = caminho_modelo_local
else:
    print("Modelo local não encontrado via metadados. Utilizando cache da Hugging Face...")
    MODELO_EMBEDDING = 'paraphrase-multilingual-MiniLM-L12-v2'

embedding_model = SentenceTransformer(MODELO_EMBEDDING, device=device)

VETORES_TOPICOS_PATH = BASE_DIR / "chatbot" / "artifacts" / "embeddings_topicos.npy"
VETORES_CHAMADOS_PATH = BASE_DIR / "chatbot" / "artifacts" / "embeddings_chamados.npy"

# Garante que a pasta de destino exista
VETORES_TOPICOS_PATH.parent.mkdir(parents=True, exist_ok=True)

print("\n--- Processando Nível 1: Tópicos ---")
if VETORES_TOPICOS_PATH.exists():
    print("Carregando vetores dos TÓPICOS salvos no disco...")
    embeddings_topicos = np.load(VETORES_TOPICOS_PATH)
else:
    print("Calculando embeddings dos TÓPICOS pela primeira vez...")
    embeddings_topicos = embedding_model.encode(
        df_topicos['texto_para_embedding'].tolist(),
        show_progress_bar=True,
        batch_size=32,
        normalize_embeddings=True
    ).astype(np.float32)
    np.save(VETORES_TOPICOS_PATH, embeddings_topicos)
    print("-> Vetores dos TÓPICOS salvos com sucesso!")

print(f"Shape Tópicos: {embeddings_topicos.shape}")


print("\n--- Processando Nível 2: Chamados ---")
if VETORES_CHAMADOS_PATH.exists():
    print("Carregando vetores dos CHAMADOS salvos no disco...")
    embeddings_chamados = np.load(VETORES_CHAMADOS_PATH)
else:
    print("Calculando embeddings dos CHAMADOS (isso pode demorar na primeira vez)...")
    # Reaproveita criar_texto_combinado da Seção original
    df['texto_completo'] = df.apply(criar_texto_combinado, axis=1)
    
    embeddings_chamados = embedding_model.encode(
        df['texto_completo'].tolist(),
        show_progress_bar=True,
        batch_size=32,
        normalize_embeddings=True
    ).astype(np.float32)
    np.save(VETORES_CHAMADOS_PATH, embeddings_chamados)
    print("-> Vetores dos CHAMADOS salvos com sucesso!")

print(f"Shape Chamados: {embeddings_chamados.shape}")

Modelo local não encontrado via metadados. Utilizando cache da Hugging Face...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3421.27it/s]



--- Processando Nível 1: Tópicos ---
Carregando vetores dos TÓPICOS salvos no disco...
Shape Tópicos: (34, 384)

--- Processando Nível 2: Chamados ---
Carregando vetores dos CHAMADOS salvos no disco...
Shape Chamados: (40645, 384)


## 6 — Indexação com FAISS (Facebook AI Similarity Search)

FAISS é uma biblioteca para **busca eficiente de vizinhos mais próximos** em espaços de alta dimensão e adicionar os embbedings dentro do Faiss.

In [7]:
from collections import defaultdict

mapa_topico_para_indices = defaultdict(list)

for idx, valor_topico in enumerate(df['topico_id_unico']):
    if pd.isna(valor_topico) or valor_topico == "NAO_CLASSIFICADO":
        continue
    
    # Se houver mais de um tópico, separa por vírgula
    lista_topicos = str(valor_topico).split(",")
    
    for t in lista_topicos:
        mapa_topico_para_indices[t.strip()].append(idx)

# Converte as listas para arrays do NumPy (necessário para o FAISS)
mapa_topico_para_indices = {k: np.array(v) for k, v in mapa_topico_para_indices.items()}

print(f"Mapeamento criado com sucesso para {len(mapa_topico_para_indices)} tópicos únicos.")

indice_topicos = faiss.IndexFlatIP(embeddings_topicos.shape[1])
indice_topicos.add(embeddings_topicos)
print(f"Indice de topicos: {indice_topicos.ntotal} vetores")

# ── Índice grande: chamados (mantido como já estava) ───────
indice_faiss = faiss.IndexFlatIP(embeddings_chamados.shape[1])
indice_faiss.add(embeddings_chamados)
print(f"Indice de chamados: {indice_faiss.ntotal} vetores")

# Mapeamento rápido: topico_id_unico -> posicoes no df de chamados
# Evita refazer esse agrupamento a cada consulta
df = pd.read_csv('../data/chatbot/df_chamados.csv')

print(f"Topicos carregados : {len(df_topicos)} (de {df_topicos['sistema_origem'].nunique()} sistemas)")
print(f"Chamados carregados: {len(df)}")

topicos_validos = set(df_topicos['topico_id_unico'])

def checar_se_tem_orfao(tags):
    if pd.isna(tags) or tags == "NAO_CLASSIFICADO":
        return True
    lista_tags = str(tags).split(",")
    return not any(t in topicos_validos for t in lista_tags)

chamados_orfaos = df[df['topico_id_unico'].apply(checar_se_tem_orfao)]

if len(chamados_orfaos) > 0:
    print(f"\nALERTA: {len(chamados_orfaos)} chamados com topico_id_unico não mapeado ou não classificado.")
else:
    print(f"\nIntegridade OK: todos os chamados mapeiam para um tópico válido!")

Mapeamento criado com sucesso para 34 tópicos únicos.
Indice de topicos: 34 vetores
Indice de chamados: 40645 vetores
Topicos carregados : 34 (de 5 sistemas)
Chamados carregados: 40645

ALERTA: 13 chamados com topico_id_unico não mapeado ou não classificado.


## 7 — Função de Busca Semântica
1. Recebe texto da pergunta
2. Gera embedding da pergunta
3. Busca os K vizinhos mais próximos no FAISS
4. Retorna os chamados mais similares com score de similaridade

In [8]:
def buscar_topicos_relevantes(pergunta: str, top_k: int = 3) -> list:
    emb_pergunta = embedding_model.encode(
        [pergunta], normalize_embeddings=True
    ).astype(np.float32)

    scores, indices = indice_topicos.search(emb_pergunta, top_k)

    resultados = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        topico = df_topicos.iloc[idx].to_dict()
        topico['score_topico'] = float(score)
        resultados.append(topico)

    return resultados


def aplicar_prioridade_sistema(topicos_recuperados: list) -> list:
    """
    Regra de desempate: quando um topico de sistema especifico
    e um topico 'Geral' aparecem juntos cobrindo o mesmo assunto,
    o sistema especifico tem prioridade.

    Implementacao: remove topicos 'Geral' do conjunto SE ja existir
    pelo menos um topico de sistema especifico com score competitivo
    (dentro de uma margem de tolerancia).
    """
    especificos = [t for t in topicos_recuperados if t['sistema_origem'] != 'Geral']
    gerais      = [t for t in topicos_recuperados if t['sistema_origem'] == 'Geral']

    if not especificos:
        # Nenhum topico especifico no top-k, mantem os gerais
        return topicos_recuperados

    if not gerais:
        return especificos

    # Tolerancia: um topico geral só sobrevive se for CLARAMENTE
    # mais relevante que o melhor especifico (margem de 10%)
    melhor_especifico = max(t['score_topico'] for t in especificos)
    gerais_relevantes  = [
        t for t in gerais
        if t['score_topico'] > melhor_especifico + 0.10
    ]

    return especificos + gerais_relevantes


def buscar_chamados_hierarquico(
    pergunta:        str,
    top_k_topicos:   int = 3,
    top_k_chamados:  int = 5,
    score_min_topico: float = 0.30
) -> dict:
    """
    Pipeline completo de busca hierárquica:
    1. Encontra tópicos relevantes (índice pequeno)
    2. Aplica regra de prioridade (sistema especifico > geral)
    3. Filtra chamados pertencentes a esses tópicos
    4. Busca semântica fina dentro do subconjunto filtrado
    """

    # Estágio 1 — tópicos
    topicos_brutos = buscar_topicos_relevantes(pergunta, top_k=top_k_topicos)
    topicos_brutos = [t for t in topicos_brutos if t['score_topico'] >= score_min_topico]

    if not topicos_brutos:
        return {
            "chamados":        [],
            "topicos_usados":  [],
            "sistema_detectado": None
        }

    # Estágio 2 — regra de prioridade
    topicos_finais = aplicar_prioridade_sistema(topicos_brutos)

    # Estágio 3 — filtragem do subconjunto
    ids_topicos = [t['topico_id_unico'] for t in topicos_finais]
    
    arrays_de_indices = [
        mapa_topico_para_indices[tid] for tid in ids_topicos
        if tid in mapa_topico_para_indices
    ]

    if not arrays_de_indices:
        return {
            "chamados":        [],
            "topicos_usados":  topicos_finais,
            "sistema_detectado": topicos_finais[0]['sistema_origem']
        }

    indices_subconjunto = np.unique(np.concatenate(arrays_de_indices))
    
    # Estágio 4 — índice temporário só do subconjunto (rápido, poucos vetores)
    embs_subconjunto = embeddings_chamados[indices_subconjunto]
    indice_temp = faiss.IndexFlatIP(embs_subconjunto.shape[1])
    indice_temp.add(embs_subconjunto)

    emb_pergunta = embedding_model.encode(
        [pergunta], normalize_embeddings=True
    ).astype(np.float32)

    k_busca = min(top_k_chamados, len(indices_subconjunto))
    scores, pos_locais = indice_temp.search(emb_pergunta, k_busca)

    chamados_resultado = []
    for score, pos_local in zip(scores[0], pos_locais[0]):
        if pos_local == -1:
            continue
        idx_global = indices_subconjunto[pos_local]
        chamado = df.iloc[idx_global].to_dict()
        chamado['score_similaridade'] = float(score)
        chamados_resultado.append(chamado)

    # Sistema detectado = o do topico com maior score
    sistema_detectado = max(topicos_finais, key=lambda t: t['score_topico'])['sistema_origem']

    return {
        "chamados":          chamados_resultado,
        "topicos_usados":    topicos_finais,
        "sistema_detectado": sistema_detectado,
        "n_chamados_filtrados": len(indices_subconjunto)
    }


# ── Teste rápido ────────────────────────────────────────────
print("Teste da busca hierarquica:")
print("=" * 60)

resultado_teste = buscar_chamados_hierarquico("Esse é o terceiro chamado que abro estou abrindo chamado pois não sei se cada caso deverá ser avaliado individualmente mas trata-se do mesmo problema: Erro ao lançar atestado no siass o número do registro informado não foi encontrado ou não está ativo.")

print(f"Sistema detectado: {resultado_teste['sistema_detectado']}")
print(f"Topicos usados: {[t['topico_id_unico'] for t in resultado_teste['topicos_usados']]}")
print(f"Chamados filtrados antes da busca fina: {resultado_teste.get('n_chamados_filtrados', 0)}")
print(f"\nChamados encontrados:")
for c in resultado_teste['chamados']:
    print(f"  [{c['score_similaridade']:.1%}] {c['Título']}")

Teste da busca hierarquica:
Sistema detectado: SIAPE
Topicos usados: ['SIAPE_2', 'TOTAIS_3', 'SIASS_0']
Chamados filtrados antes da busca fina: 3076

Chamados encontrados:
  [77.6%] erro ao lancar atestado no siass o numero do registro informado nao foi encontrado ou nao esta ativo
  [58.1%] desfazimento de registro de atestado com fundamento na medida provisoria no
  [55.1%] s erro
  [53.1%] erro
  [52.3%] erros associados e


## 8 — Modelo LLM Qwen

### Modelo escolhido: `Qwen/Qwen2.5-3B-Instruct`




In [9]:
import os
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

BASE_DIR = Path.cwd().parent
PASTA_DESTINO = BASE_DIR / "chatbot" / "models" / "llm" / "Qwen__Qwen2.5-3B-Instruct"

os.makedirs(PASTA_DESTINO, exist_ok=True)

if (PASTA_DESTINO / "config.json").exists():
    print(f"Modelo e Tokenizer já existem localmente em:\n{PASTA_DESTINO}")
    print("Carregando os arquivos do disco local...")
    
    tokenizer_llm = AutoTokenizer.from_pretrained(PASTA_DESTINO)
    modelo_llm = AutoModelForCausalLM.from_pretrained(PASTA_DESTINO)
else:
    NOME_OFICIAL_MODELO = "Qwen/Qwen2.5-3B-Instruct"
    print(f"Baixando o Tokenizer e o Modelo de '{NOME_OFICIAL_MODELO}'...")

    tokenizer_llm = AutoTokenizer.from_pretrained(NOME_OFICIAL_MODELO)
    modelo_llm = AutoModelForCausalLM.from_pretrained(NOME_OFICIAL_MODELO)

    print("Download concluído! Salvando os arquivos no disco local...")

    tokenizer_llm.save_pretrained(PASTA_DESTINO)
    modelo_llm.save_pretrained(PASTA_DESTINO, safe_serialization=True)

    print(f"✅ Sucesso! Modelo e Tokenizer salvos definitivamente em:\n{PASTA_DESTINO}")

modelo_llm = modelo_llm.to(device)
modelo_llm.eval()

def gerar_resposta_llm(prompt: str, max_tokens: int = 800) -> str:
    messages = [
        {"role": "system", "content": "Você é um assistente especializado em suporte técnico de TI. Responda sempre em português do Brasil de forma clara e objetiva."},
        {"role": "user", "content": prompt}
    ]
    
    texto = tokenizer_llm.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = tokenizer_llm(texto, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = modelo_llm.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.2,        
            do_sample=False,        
            num_beams=1,
            repetition_penalty=1.1,
        )

    resposta = tokenizer_llm.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    
    return resposta.strip()

Modelo e Tokenizer já existem localmente em:
C:\Users\anacl\OneDrive\Documentos\IC\Chamados-MGI\chatbot\models\llm\Qwen__Qwen2.5-3B-Instruct
Carregando os arquivos do disco local...


Loading weights: 100%|██████████| 434/434 [00:00<00:00, 802.16it/s] 


## 9 - Pipeline RAG - busca + ia generativa

In [11]:
import time
import sys

def imprimir_devagar(texto: str, delay: float = 0.018):
    """
    Imprime o texto caractere por caractere para uma leitura agradável.
    delay: tempo em segundos entre cada caractere (0.018 = ~55 chars/seg)
    """
    for char in texto:
        sys.stdout.write(char)
        sys.stdout.flush()
        time.sleep(delay)
    print()


def construir_prompt(pergunta: str, chamados_contexto: list) -> str:

    SCORE_SOLUCAO_DIRETA  = 0.75
    SCORE_SOLUCAO_PARCIAL = 0.60

    contexto_partes = []
    for i, chamado in enumerate(chamados_contexto, 1):
        bloco = (
            f"Chamado {i} (similaridade: {chamado['score_similaridade']:.0%}):\n"
            f"  Titulo: {chamado['Título']}\n"
            f"  Descricao: {chamado['Descrição do chamado']}\n"
            f"  Ultima acao aplicada: {chamado['Última ação de acompanhamento']}\n"
            f"  Equipe responsavel: {chamado['Time']}\n"
        )
        contexto_partes.append(bloco)

    contexto_texto = "\n\n".join(contexto_partes)
    melhor_score   = chamados_contexto[0]['score_similaridade'] if chamados_contexto else 0.0
    melhor_chamado = chamados_contexto[0] if chamados_contexto else None
    texto_solucao_topo = str(melhor_chamado.get('Última ação de acompanhamento', '')) if melhor_chamado else ""
    link_util = extrair_link_da_solucao(texto_solucao_topo)

    bloco_link = ""
    if link_util:
        bloco_link = f"""
=== LINK OBRIGATÓRIO PARA DIRECIONAMENTO ===
Foi identificado um link oficial para a resolução deste problema.
Você DEVE incluir o link abaixo na sua resposta e orientar o usuário a acessá-lo.
Link oficial: {link_util}
"""

    if melhor_score >= SCORE_SOLUCAO_DIRETA:
        instrucao_fluxo = f"""
O score de similaridade e ALTO ({melhor_score:.0%}).
Ha um chamado resolvido muito parecido com o problema relatado.

Responda OBRIGATORIAMENTE iniciando com a linha:
SOLUCAO ENCONTRADA

Em seguida responda com estas secoes, sem emojis, sem markdown:

O que esta acontecendo:
[Explique em 2 linhas o que provavelmente esta ocorrendo com base no chamado similar]

O que voce pode tentar agora:
[Liste de 2 a 4 passos praticos e simples, baseados diretamente na "Ultima acao aplicada" do chamado similar.
 Escreva de forma que um usuario nao tecnico consiga executar sem dificuldade.
 Exemplo: "1. Acesse o portal em...", "2. Clique em...", "3. Caso o erro persista, tente..."]

Se os passos acima nao resolverem, sera necessario abrir um chamado.

Titulo sugerido para o chamado:
[Escreva um titulo claro com ate 10 palavras descrevendo o problema]

Descricao sugerida para o chamado:
[Escreva uma descricao completa de 3 a 5 linhas com: o que esta acontecendo,
 quando comeou ou em que situacao ocorre, quais sistemas ou usuarios sao afetados
 e o que ja foi tentado]
"""

    elif melhor_score >= SCORE_SOLUCAO_PARCIAL:
        instrucao_fluxo = f"""
O score de similaridade e MEDIO ({melhor_score:.0%}).
Ha chamados relacionados mas o problema pode ter variacoes.

Responda OBRIGATORIAMENTE iniciando com a linha:
CHAMADO RECOMENDADO

Em seguida responda com estas secoes, sem emojis, sem markdown:

O que pode estar acontecendo:
[Explique em 2 linhas com base nos chamados similares.
 Deixe claro que o problema pode ter variacoes e que o atendimento tecnico e necessario]

O que voce pode tentar enquanto aguarda:
[Liste 1 a 2 passos simples e sem risco que o usuario pode tentar por conta propria]
Se houver link

Titulo sugerido para o chamado:
[Escreva um titulo claro com ate 10 palavras]

Descricao sugerida para o chamado:
[Escreva uma descricao completa de 3 a 5 linhas com: o que esta acontecendo,
 quando comeou, quais sistemas ou usuarios sao afetados e o que ja foi tentado]

Informacoes importantes para incluir no chamado:
- [Dado relevante 1 que o tecnico vai precisar saber]
- [Dado relevante 2 que o tecnico vai precisar saber]
"""

    else:
        instrucao_fluxo = f"""
O score de similaridade e BAIXO ({melhor_score:.0%}).
Nao ha chamados similares suficientes para sugerir uma solucao direta.

Analise cuidadosamente a mensagem do usuario e escolha UMA das opcoes abaixo:

OPCAO A — se for uma duvida simples e informativa (exemplo: como acessar
um sistema, onde encontrar algo, procedimento padrao):

Responda OBRIGATORIAMENTE iniciando com a linha:
RESPOSTA DIRETA

Em seguida responda a duvida de forma clara e objetiva em ate 4 linhas,
usando linguagem simples acessivel para usuarios nao tecnicos.

---

OPCAO B — se for um problema tecnico que exige intervencao da equipe:

Responda OBRIGATORIAMENTE iniciando com a linha:
CHAMADO NECESSARIO

Em seguida responda com estas secoes, sem emojis, sem markdown:

Por que e necessario abrir um chamado:
[Explique em 2 linhas por que esse problema precisa de atendimento tecnico]

Titulo sugerido para o chamado:
[Escreva um titulo claro com ate 10 palavras]

Descricao sugerida para o chamado:
[Escreva uma descricao completa de 3 a 5 linhas com: o que esta acontecendo,
 quando comeou e quais sistemas ou usuarios sao afetados]

Informacoes importantes para incluir no chamado:
- [Dado relevante 1]
- [Dado relevante 2]
"""

    prompt = f"""Voce e um assistente virtual de suporte tecnico do Portal MGI.
Seu publico sao colaboradores que NAO sao tecnicos de TI.
Responda SEMPRE em portugues do Brasil, de forma clara, acolhedora e acessivel.
Nunca use emojis na resposta.

=== REGRAS CRÍTICAS DE SEGURANÇA ===
1. NUNCA invente, crie ou altere URLs, links ou sites.
2. Se a seção "LINK OBRIGATÓRIO PARA DIRECIONAMENTO" existir abaixo, use EXATAMENTE o link fornecido nela.
3. Se não houver link explícito fornecido por mim, NÃO adicione nenhum link na sua resposta.

=== CHAMADOS HISTORICOS SIMILARES ===
{contexto_texto}
{bloco_link}

=== INSTRUCAO ===
{instrucao_fluxo}

=== MENSAGEM DO USUARIO ===
{pergunta}

Resposta:"""

    return prompt

def chatbot_rag(pergunta: str, top_k: int = 10, verbose: bool = False) -> dict:

    inicio = time.time()
    chamados = buscar_chamados_hierarquico(pergunta, top_k_chamados=top_k)
    melhor_score = chamados[0]['score_similaridade'] if chamados else 0.0

    if not chamados:
        return {
            "resposta": (
                "CHAMADO NECESSARIO\n\n"
                "Nao encontrei registros similares na base de chamados.\n\n"
                "Para garantir o atendimento, abra um chamado descrevendo:\n"
                "  - O que esta acontecendo\n"
                "  - Desde quando o problema ocorre\n"
                "  - Quais usuarios ou sistemas sao afetados"
            ),
            "fluxo_detectado": "sem_contexto",
            "melhor_score": 0.0,
            "chamados_recuperados": [],
            "tempo_s": time.time() - inicio
        }

    prompt = construir_prompt(pergunta, chamados)
    resposta = gerar_resposta_llm(prompt, max_tokens=400)

    primeira_linha = resposta.strip().split('\n')[0].upper()

    if "SOLUCAO ENCONTRADA" in primeira_linha:
        fluxo = "solucao_disponivel"
    elif "CHAMADO RECOMENDADO" in primeira_linha:
        fluxo = "chamado_recomendado"
    elif "RESPOSTA DIRETA" in primeira_linha:
        fluxo = "faq_direto"
    elif "CHAMADO NECESSARIO" in primeira_linha:
        fluxo = "novo_chamado"
    else:
        if melhor_score >= 0.75:
            fluxo = "solucao_disponivel"
        elif melhor_score >= 0.60:
            fluxo = "chamado_recomendado"
        else:
            fluxo = "novo_chamado"

    return {
        "resposta": resposta,
        "fluxo_detectado": fluxo,
        "melhor_score": melhor_score,
        "chamados_recuperados": chamados,
        "tempo_s": time.time() - inicio,
        "prompt_usado": prompt
    }

## 10 - Interface de chat interativa

Agora criamos um loop de chat para interagir com o chatbot de forma contínua, com histórico da conversa.

**Para encerrar**: digite `sair`, `exit` ou `quit`

In [1]:
def chatbot_rag(pergunta: str, top_k: int = 10, verbose: bool = False) -> dict:

     inicio = time.time()
     resultado_busca = buscar_chamados_hierarquico(pergunta, top_k_chamados=top_k)
     chamados = resultado_busca.get("chamados", [])
     melhor_score = chamados[0]['score_similaridade'] if chamados else 0.0

     prompt = construir_prompt(pergunta, chamados)
     resposta = gerar_resposta_llm(prompt)

     primeira_linha = resposta.strip().split('\n')[0].upper()

     if "SOLUCAO ENCONTRADA" in primeira_linha:
         fluxo = "solucao_disponivel"
     elif "CHAMADO RECOMENDADO" in primeira_linha:
         fluxo = "chamado_recomendado"
     elif "RESPOSTA DIRETA" in primeira_linha:
         fluxo = "faq_direto"
     elif "CHAMADO NECESSARIO" in primeira_linha:
         fluxo = "novo_chamado"
     else:
         if melhor_score >= 0.75:
             fluxo = "solucao_disponivel"
         elif melhor_score >= 0.60:
             fluxo = "chamado_recomendado"
         else:
             fluxo = "novo_chamado"

     return {
         "resposta": resposta,
         "fluxo_detectado": fluxo,
         "melhor_score": melhor_score,
         "chamados_recuperados": chamados,
         "tempo_s": time.time() - inicio,
         "prompt_usado": prompt
     }

def exibir_resultado_formatado(resultado: dict):

     labels = {
         "faq_direto":          "Resposta direta",
         "solucao_disponivel":  "Solucao encontrada no historico",
         "chamado_recomendado": "Chamado recomendado",
         "novo_chamado":        "Abertura de chamado necessaria",
         "sem_contexto":        "Sem historico similar",
         "indefinido":          "Fluxo nao identificado",
     }

     fluxo = resultado.get("fluxo_detectado", "indefinido")

     print("\n" + "=" * 65)
     print("ASSISTENTE DE SUPORTE TECNICO")
     print("=" * 65)
     print(f"Fluxo    : {labels.get(fluxo, fluxo)}")
     print(f"Score    : {resultado.get('melhor_score', 0):.1%}")
     print("-" * 65 + "\n")

     imprimir_devagar(resultado['resposta'])

     print(f"\n" + "-" * 65)
     print(f"Tempo de resposta: {resultado['tempo_s']:.1f}s")
     print("=" * 65)


def chatbot_interativo():

     historico  = []
     contadores = {
         "faq_direto":          0,
         "solucao_disponivel":  0,
         "chamado_recomendado": 0,
         "novo_chamado":        0,
         "sem_contexto":        0,
         "indefinido":          0,
     }
     print("  ASSISTENTE DE SUPORTE TECNICO — FAQ INTELIGENTE")
     print("""
   Como posso te ajudar?

   Voce pode descrever um problema tecnico, relatar um erro
   ou fazer uma pergunta sobre os sistemas do portal.

   O assistente ira:
     - Responder diretamente se for uma duvida simples
     - Sugerir uma solucao se houver chamado similar no historico
     - Orientar a abertura de chamado quando necessario,
       com titulo e descricao ja sugeridos

   Obs: Evite compartilhar dados sensíveis, descreva o seu problema de forma anônima e sem valores de matricula ou CPF.
   Não confie cegamente nesse chatbot, é uma ferramenta de auxilio na abertura de chamados e tira duvidas.

   Comandos disponiveis: historico | stats | sair
   """)

     while True:

         try:
             pergunta = input("\nVocê: ").strip()
         except EOFError:
             print("\n(Sessao encerrada)")
             break

         if not pergunta:
             print("  Por favor, descreva sua duvida ou problema.")
             continue

         if pergunta.lower() in ['sair', 'exit', 'quit']:
             print(f"\nEncerrando sessao.")
             print(f"Total de interacoes: {len(historico)}")
             if historico:
                 print("Fluxos acionados durante a sessao:")
                 labels = {
                     "faq_direto":          "Resposta direta",
                     "solucao_disponivel":  "Solucao encontrada",
                     "chamado_recomendado": "Chamado recomendado",
                     "novo_chamado":        "Novo chamado",
                     "sem_contexto":        "Sem contexto",
                     "indefinido":          "Indefinido",
                 }
                 for fluxo, count in contadores.items():
                     if count > 0:
                         print(f"  {labels.get(fluxo, fluxo)}: {count}x")
             print("\nAte mais.\n")
             break

         if pergunta.lower() == 'historico':
             if not historico:
                 print("\n  Nenhuma interacao registrada ainda.")
             else:
                 print(f"\n  Historico da sessao ({len(historico)} interacao(oes)):")
                 print("  " + "-" * 55)
                 labels_curtos = {
                     "faq_direto":          "[FAQ]     ",
                     "solucao_disponivel":  "[SOLUCAO] ",
                     "chamado_recomendado": "[PARCIAL] ",
                     "novo_chamado":        "[CHAMADO] ",
                     "sem_contexto":        "[SEM CTX] ",
                     "indefinido":          "[?]       ",
                 }
                 for i, h in enumerate(historico, 1):
                     label = labels_curtos.get(h['fluxo'], "[?]")
                     trecho = h['pergunta'][:50]
                     reticencias = '...' if len(h['pergunta']) > 50 else ''
                     print(f"  {i:>2}. {label} [{h['score']:.0%}] {trecho}{reticencias}")
             continue

         if pergunta.lower() == 'stats':
             total = len(historico)
             if total == 0:
                 print("\n  Nenhuma interacao registrada ainda.")
             else:
                 print(f"\n  Estatisticas da sessao ({total} interacao(oes)):")
                 print("  " + "-" * 55)
                 labels = {
                     "faq_direto":          "Respondidas diretamente",
                     "solucao_disponivel":  "Solucao encontrada",
                     "chamado_recomendado": "Chamado recomendado",
                     "novo_chamado":        "Novo chamado necessario",
                     "sem_contexto":        "Sem contexto similar",
                     "indefinido":          "Fluxo indefinido",
                 }
                 for fluxo, count in contadores.items():
                     if count > 0:
                         pct   = count / total * 100
                         barra = "#" * int(pct / 5) + "-" * (20 - int(pct / 5))
                         print(f"  {labels.get(fluxo, fluxo):<28} "
                               f"{barra} {count}x ({pct:.0f}%)")
                 scores = [h['score'] for h in historico]
                 print(f"\n  Score medio de similaridade: {sum(scores)/len(scores):.1%}")
             continue

         print("\n  Analisando sua mensagem...")
         resultado = chatbot_rag(pergunta, top_k=3, verbose=False)
         exibir_resultado_formatado(resultado)

         fluxo = resultado.get("fluxo_detectado", "indefinido")
         historico.append({
             "pergunta": pergunta,
             "fluxo":    fluxo,
             "score":    resultado.get("melhor_score", 0.0),
             "resposta": resultado["resposta"],
         })
         contadores[fluxo] = contadores.get(fluxo, 0) + 1

chatbot_interativo()

  ASSISTENTE DE SUPORTE TECNICO — FAQ INTELIGENTE

   Como posso te ajudar?

   Voce pode descrever um problema tecnico, relatar um erro
   ou fazer uma pergunta sobre os sistemas do portal.

   O assistente ira:
     - Responder diretamente se for uma duvida simples
     - Sugerir uma solucao se houver chamado similar no historico
     - Orientar a abertura de chamado quando necessario,
       com titulo e descricao ja sugeridos

   Obs: Evite compartilhar dados sensíveis, descreva o seu problema de forma anônima e sem valores de matricula ou CPF.
   Não confie cegamente nesse chatbot, é uma ferramenta de auxilio na abertura de chamados e tira duvidas.

   Comandos disponiveis: historico | stats | sair
   

  Analisando sua mensagem...


NameError: name 'time' is not defined

## 11 - Interface de chat interativa (Suporte técnico)

Agora criamos um loop de chat para interagir com o chatbot de forma contínua, com histórico da conversa.

**Para encerrar**: digite `sair`, `exit` ou `quit`

In [15]:
import time
import sys

SCORE_ALTO   = 0.70
SCORE_MEDIO  = 0.50



def buscar_chamados_similares_suporte(
    pergunta:     str,
    top_k:        int   = 10,
    filtro_time:  str   = None,
    score_minimo: float = 0.40,
    top_k_topicos: int  = 3
) -> list:
    """
    Versao hierarquica da busca do analista.
    Reaproveita buscar_chamados_hierarquico (Secao 7) como primeiro estagio,
    depois aplica o filtro_time por cima do resultado ja filtrado por topico.
    """

    resultado_hier = buscar_chamados_hierarquico(
        pergunta,
        top_k_topicos=top_k_topicos,
        top_k_chamados=top_k * 3 if filtro_time else top_k,
        score_min_topico=0.30
    )

    chamados = resultado_hier['chamados']

    # Aplica score_minimo (o hierarquico ja filtra por topico, mas nao por score absoluto)
    chamados = [c for c in chamados if c['score_similaridade'] >= score_minimo]

    # Aplica filtro_time por cima, se informado
    if filtro_time:
        chamados = [
            c for c in chamados
            if filtro_time.upper() in str(c.get('Time', '')).upper()
        ]

    return chamados[:top_k]


def buscar_por_id(id_chamado: str) -> list:
    """
    Mantida igual — usa o titulo do chamado de referencia
    como pergunta para o pipeline hierarquico.
    """
    mascara = df['Id'].astype(str) == str(id_chamado).strip()

    if not mascara.any():
        return []

    chamado_ref = df[mascara].iloc[0]

    texto_ref = (
        f"{chamado_ref['Título']} "
        f"{chamado_ref['Descrição do chamado']}"
    )

    return buscar_chamados_similares_suporte(texto_ref, top_k=6)


def analisar_padrao(chamados: list) -> dict:
    """
    Analisa os chamados recuperados e extrai padroes:
    - Time mais frequente
    - Tempo medio de resolucao
    - Se ha concentracao de solucoes similares
    """
    if not chamados:
        return {}

    times      = [c.get('Time', 'N/A') for c in chamados]
    tempos     = [c.get('Tempo em horas do atendimento', 0) for c in chamados]
    scores     = [c.get('score_similaridade', 0) for c in chamados]
    categorias = [c.get('Categoria', 'N/A') for c in chamados]

    time_mais_freq  = max(set(times), key=times.count)
    freq_time       = times.count(time_mais_freq)
    categoria_freq  = max(set(categorias), key=categorias.count)
    tempos_validos  = [t for t in tempos if t and t > 0]
    tempo_medio     = np.mean(tempos_validos) if tempos_validos else 0
    score_medio     = np.mean(scores)
    concentrado     = freq_time >= len(chamados) * 0.6  # 60%+ no mesmo time

    return {
        "time_mais_frequente":  time_mais_freq,
        "frequencia_time":      freq_time,
        "total_chamados":       len(chamados),
        "categoria_frequente":  categoria_freq,
        "tempo_medio_h":        tempo_medio,
        "score_medio":          score_medio,
        "padrao_concentrado":   concentrado,
    }


def construir_prompt_suporte(pergunta: str, chamados_contexto: list) -> str:
    """
    Prompt para o analista de suporte.
    Linguagem tecnica, densa, sem simplificacoes.
    Exibe IDs, categorias, acoes anteriores e sugere escalonamento.
    """

    melhor_score   = chamados_contexto[0]['score_similaridade'] if chamados_contexto else 0.0
    melhor_chamado = chamados_contexto[0] if chamados_contexto else None
    padrao         = analisar_padrao(chamados_contexto)

    contexto_partes = []
    for i, c in enumerate(chamados_contexto, 1):
        bloco = (
            f"Chamado {i} | ID: {c.get('Id', 'N/A')} | Score: {c['score_similaridade']:.0%}\n"
            f"  Titulo     : {c.get('Título', 'N/A')}\n"
            f"  Categoria  : {c.get('Categoria', 'N/A')}\n"
            f"  Time       : {c.get('Time', 'N/A')}\n"
            f"  Status     : {c.get('Status', 'N/A')}\n"
            f"  Unidade    : {c.get('Unidade', 'N/A')}\n"
            f"  Tempo res. : {c.get('Tempo em horas do atendimento', 'N/A')}h\n"
            f"  Descricao  : {str(c.get('Descrição do chamado', ''))[:300]}\n"
            f"  Ultima acao: {str(c.get('Última ação de acompanhamento', ''))[:300]}"
        )
        contexto_partes.append(bloco)

    contexto_texto = "\n\n".join(contexto_partes)

    if padrao:
        resumo_padrao = (
            f"Time mais recorrente : {padrao['time_mais_frequente']} "
            f"({padrao['frequencia_time']}/{padrao['total_chamados']} chamados)\n"
            f"Categoria predominante: {padrao['categoria_frequente']}\n"
            f"Tempo medio historico : {padrao['tempo_medio_h']:.1f}h\n"
            f"Score medio           : {padrao['score_medio']:.0%}\n"
            f"Padrao concentrado    : {'Sim' if padrao['padrao_concentrado'] else 'Nao'}"
        )
    else:
        resumo_padrao = "Sem dados suficientes para analise de padrao."

    if melhor_score >= SCORE_ALTO:
        instrucao = f"""
O score de similaridade e ALTO ({melhor_score:.0%}).
Ha um padrao claro identificado nos chamados historicos.

Responda OBRIGATORIAMENTE iniciando com a linha exata:
PADRAO IDENTIFICADO

Em seguida, responda com as secoes abaixo sem emojis e sem markdown:

Diagnostico tecnico:
[Com base nos chamados similares, descreva tecnicamente o que esta ocorrendo,
 qual o componente ou processo afetado e qual a causa mais provavel]

Acoes recomendadas:
[Liste de 3 a 5 acoes tecnicas especificas baseadas nas ultimas acoes dos chamados historicos.
 Seja direto e tecnico. Inclua comandos, caminhos de menu ou parametros relevantes se houver.]

Escalonamento:
[Indique se o padrao historico sugere escalonamento para outro time.
 Se sim, qual time e por que. Se nao, confirme que o time atual e adequado.]

Alertas:
[Aponte qualquer anomalia nos dados — tempo de resolucao fora do padrao,
 chamados recorrentes da mesma unidade, ou qualquer sinal de problema sistemico]
"""

    elif melhor_score >= SCORE_MEDIO:
        instrucao = f"""
O score de similaridade e MEDIO ({melhor_score:.0%}).
Os chamados sao relacionados mas o problema pode ter variacoes.

Responda OBRIGATORIAMENTE iniciando com a linha exata:
ANALISE NECESSARIA

Em seguida, responda com as secoes abaixo sem emojis e sem markdown:

Hipoteses de diagnostico:
[Liste de 2 a 3 hipoteses tecnicas para o problema, baseadas nos chamados similares.
 Ordene da mais para a menos provavel com base nos scores de similaridade.]

Investigacao recomendada:
[Quais informacoes adicionais o analista deve coletar do solicitante para
 confirmar o diagnostico? Liste de forma objetiva.]

Acoes iniciais possiveis:
[Acoes que podem ser iniciadas antes da confirmacao do diagnostico,
 sem risco de causar impacto negativo]

Escalonamento provavel:
[Com base nos chamados similares, qual time tem mais chance de ser o responsavel
 pela resolucao final? Justifique.]
"""

    else:
        instrucao = f"""
O score de similaridade e BAIXO ({melhor_score:.0%}).
Problema sem precedente claro na base historica.

Responda OBRIGATORIAMENTE iniciando com a linha exata:
CASO INEDITO

Em seguida, responda com as secoes abaixo sem emojis e sem markdown:

Avaliacao inicial:
[Com base apenas na descricao do problema, faca uma avaliacao tecnica inicial
 do que pode estar ocorrendo, deixando claro que nao ha historico similar]

Informacoes criticas a coletar:
[Liste de 4 a 6 informacoes que o analista deve obter do solicitante antes
 de qualquer acao. Seja especifico sobre logs, versoes, ambiente, etc.]

Time sugerido para triagem:
[Com base no tipo de problema descrito, qual time tem mais chance de ser
 o responsavel, mesmo sem historico confirmando isso?]

Recomendacao:
[Abertura formal de chamado com prioridade sugerida: baixa, media, alta ou critica,
 com justificativa tecnica]
"""

    prompt = f"""Voce e um assistente tecnico especializado para analistas de suporte do Portal MGI.
Seu publico sao tecnicos experientes. Use linguagem tecnica, seja direto e denso.
Nao simplifique. Nao use emojis. Responda sempre em portugues do Brasil.
Baseie suas respostas exclusivamente nos chamados historicos fornecidos.
Nao invente dados, IDs ou solucoes que nao estejam no contexto.

=== ANALISE DE PADRAO DOS CHAMADOS RECUPERADOS ===
{resumo_padrao}

=== CHAMADOS HISTORICOS SIMILARES ===
{contexto_texto}

=== PROBLEMA RELATADO PELO ANALISTA ===
{pergunta}

=== INSTRUCAO ===
{instrucao}

Resposta:"""

    return prompt


def chatbot_rag_suporte(
    pergunta:    str,
    top_k:       int  = 7,
    filtro_time: str  = None,
    verbose:     bool = False
) -> dict:

    inicio = time.time()

    chamados = buscar_chamados_similares_suporte(
        pergunta, top_k=top_k, filtro_time=filtro_time
    )
    melhor_score = chamados[0]['score_similaridade'] if chamados else 0.0
    padrao       = analisar_padrao(chamados)

    if not chamados:
        return {
            "resposta": (
                "CASO INEDITO\n\n"
                "Nenhum chamado similar encontrado na base.\n"
                "Recomenda-se abertura de chamado com descricao detalhada\n"
                "e encaminhamento para triagem manual."
            ),
            "fluxo_detectado":  "inedito",
            "melhor_score":     0.0,
            "chamados_recuperados": [],
            "padrao":           {},
            "tempo_s":          time.time() - inicio
        }

    prompt   = construir_prompt_suporte(pergunta, chamados)
    resposta = gerar_resposta_llm(prompt, max_tokens=500)

    primeira_linha = resposta.strip().split('\n')[0].upper()

    if "PADRAO IDENTIFICADO" in primeira_linha:
        fluxo = "padrao_identificado"
    elif "ANALISE NECESSARIA" in primeira_linha:
        fluxo = "analise_necessaria"
    elif "CASO INEDITO" in primeira_linha:
        fluxo = "inedito"
    else:
        if melhor_score >= SCORE_ALTO:
            fluxo = "padrao_identificado"
        elif melhor_score >= SCORE_MEDIO:
            fluxo = "analise_necessaria"
        else:
            fluxo = "inedito"

    return {
        "resposta":             resposta,
        "fluxo_detectado":      fluxo,
        "melhor_score":         melhor_score,
        "chamados_recuperados": chamados,
        "padrao":               padrao,
        "tempo_s":              time.time() - inicio,
        "prompt_usado":         prompt
    }


def exibir_resultado_suporte(resultado: dict):
    """
    Exibe resultado para o analista — denso, com dados tecnicos,
    sem escrita lenta (o tecnico precisa de velocidade).
    """
    labels = {
        "padrao_identificado": "Padrao identificado",
        "analise_necessaria":  "Analise necessaria",
        "inedito":             "Caso inedito",
    }

    fluxo  = resultado.get("fluxo_detectado", "inedito")
    padrao = resultado.get("padrao", {})

    print("\n" + "=" * 70)
    print("ASSISTENTE DE SUPORTE — MODULO DO ANALISTA")
    print("=" * 70)
    print(f"Fluxo         : {labels.get(fluxo, fluxo)}")
    print(f"Score maximo  : {resultado.get('melhor_score', 0):.1%}")

    if padrao:
        print(f"Time recorrente: {padrao.get('time_mais_frequente', 'N/A')} "
              f"({padrao.get('frequencia_time', 0)}/"
              f"{padrao.get('total_chamados', 0)} chamados)")
        print(f"Tempo medio   : {padrao.get('tempo_medio_h', 0):.1f}h")

    print("-" * 70)
    print("\nRESPOSTA:\n")
    print(resultado['resposta'])

    chamados = resultado.get('chamados_recuperados', [])
    if chamados:
        print("\n" + "-" * 70)
        print(f"CHAMADOS DE REFERENCIA ({len(chamados)}):")
        print("-" * 70)
        print(f"{'Score':>6}  {'ID':>10}  {'Time':<25}  {'Titulo':<35}")
        print("-" * 70)
        for c in chamados:
            titulo = str(c.get('Título', ''))[:35]
            time_  = str(c.get('Time', ''))[:25]
            id_    = str(c.get('Id', 'N/A'))
            score  = c.get('score_similaridade', 0)
            print(f"{score:>6.0%}  {id_:>10}  {time_:<25}  {titulo:<35}")

    print(f"\nTempo de resposta: {resultado['tempo_s']:.1f}s")
    print("=" * 70)


def chatbot_interativo_suporte():
    """
    Loop do chat para o analista de suporte.

    Comandos especiais:
      /time <nome>   filtra busca por time especifico
      /id <numero>   busca chamados similares a um ID especifico
      /times         lista todos os times disponíveis na base
      historico      exibe interacoes da sessao
      stats          exibe estatisticas da sessao
      sair           encerra
    """

    historico    = []
    filtro_time  = None
    contadores   = {
        "padrao_identificado": 0,
        "analise_necessaria":  0,
        "inedito":             0,
    }

    print("\n" + "=" * 70)
    print("  ASSISTENTE DE SUPORTE TECNICO — MODULO DO ANALISTA")
    print("=" * 70)
    print("""
  Descreva o problema ou use um dos comandos abaixo:

  /time <nome>   filtra busca dentro de um time especifico
                 exemplo: /time FOLHA qual o procedimento para rubrica?
  /id <numero>   busca chamados similares a um ID existente
                 exemplo: /id 42013898
  /times         lista todos os times disponíveis
  historico      exibe interacoes desta sessao
  stats          estatisticas dos fluxos acionados
  sair           encerra a sessao
  """ + "-" * 70)

    while True:

        try:
            entrada = input("\nAnalista: ").strip()
        except EOFError:
            print("\n(Sessao encerrada)")
            break

        if not entrada:
            print("  Descreva o problema ou use um comando.")
            continue

        if entrada.lower() in ['sair', 'exit', 'quit']:
            print(f"\nSessao encerrada.")
            print(f"Total de consultas: {len(historico)}")
            if historico:
                print("Distribuicao de fluxos:")
                labels = {
                    "padrao_identificado": "Padrao identificado",
                    "analise_necessaria":  "Analise necessaria",
                    "inedito":             "Caso inedito",
                }
                for fluxo, count in contadores.items():
                    if count > 0:
                        print(f"  {labels.get(fluxo, fluxo)}: {count}x")
            print()
            break

        if entrada.lower() == '/times':
            print("\nTimes disponíveis na base:")
            print("-" * 50)
            for time_nome, count in df['Time'].value_counts().items():
                print(f"  {time_nome:<45} {count:>6} chamados")
            continue

        if entrada.lower() == 'historico':
            if not historico:
                print("\n  Nenhuma consulta registrada ainda.")
            else:
                print(f"\n  Historico da sessao ({len(historico)} consulta(s)):")
                print("  " + "-" * 60)
                labels_curtos = {
                    "padrao_identificado": "[PADRAO]  ",
                    "analise_necessaria":  "[ANALISE] ",
                    "inedito":             "[INEDITO] ",
                }
                for i, h in enumerate(historico, 1):
                    label  = labels_curtos.get(h['fluxo'], "[?]      ")
                    trecho = h['pergunta'][:55]
                    ret    = '...' if len(h['pergunta']) > 55 else ''
                    filtro = f" [/{h['filtro_time']}]" if h.get('filtro_time') else ""
                    print(f"  {i:>2}. {label} [{h['score']:.0%}]{filtro} {trecho}{ret}")
            continue

        if entrada.lower() == 'stats':
            total = len(historico)
            if total == 0:
                print("\n  Nenhuma consulta registrada ainda.")
            else:
                print(f"\n  Estatisticas da sessao ({total} consulta(s)):")
                print("  " + "-" * 60)
                labels = {
                    "padrao_identificado": "Padrao identificado",
                    "analise_necessaria":  "Analise necessaria ",
                    "inedito":             "Caso inedito       ",
                }
                for fluxo, count in contadores.items():
                    if count > 0:
                        pct   = count / total * 100
                        barra = "#" * int(pct / 5) + "-" * (20 - int(pct / 5))
                        print(f"  {labels.get(fluxo, fluxo):<22} "
                              f"{barra} {count}x ({pct:.0f}%)")
                scores = [h['score'] for h in historico]
                print(f"\n  Score medio: {np.mean(scores):.1%}")
                print(f"  Score min  : {min(scores):.1%}")
                print(f"  Score max  : {max(scores):.1%}")
            continue

        if entrada.lower().startswith('/id '):
            id_chamado = entrada[4:].strip()
            print(f"\n  Buscando chamados similares ao ID {id_chamado}...")

            chamados_similares = buscar_por_id(id_chamado)

            if not chamados_similares:
                print(f"  ID {id_chamado} nao encontrado na base.")
                continue

            ref = df[df['Id'].astype(str) == id_chamado]
            if not ref.empty:
                r = ref.iloc[0]
                print(f"\n  Chamado de referencia:")
                print(f"  Titulo   : {r['Título']}")
                print(f"  Time     : {r['Time']}")
                print(f"  Categoria: {r.get('Categoria', 'N/A')}")
                print(f"  Status   : {r.get('Status', 'N/A')}")

            pergunta_id = str(ref.iloc[0]['Título']) if not ref.empty else id_chamado
            resultado   = chatbot_rag_suporte(pergunta_id, top_k=6)
            exibir_resultado_suporte(resultado)

            fluxo = resultado.get("fluxo_detectado", "inedito")
            historico.append({
                "pergunta":    f"/id {id_chamado}",
                "fluxo":       fluxo,
                "score":       resultado.get("melhor_score", 0.0),
                "filtro_time": None,
            })
            contadores[fluxo] = contadores.get(fluxo, 0) + 1
            continue

        filtro_ativo = None
        pergunta     = entrada

        if entrada.lower().startswith('/time '):
            resto       = entrada[6:].strip()
            partes      = resto.split(' ', 1)
            filtro_ativo = partes[0]
            pergunta     = partes[1] if len(partes) > 1 else ""

            if not pergunta:
                print(f"  Filtro ativado para o time: {filtro_ativo}")
                print(f"  Agora descreva o problema.")
                filtro_time = filtro_ativo
                continue

            print(f"\n  Buscando dentro do time: {filtro_ativo}")

        print("\n  Analisando...")
        resultado = chatbot_rag_suporte(
            pergunta,
            top_k=7,
            filtro_time=filtro_ativo or filtro_time
        )
        exibir_resultado_suporte(resultado)

        fluxo = resultado.get("fluxo_detectado", "inedito")
        historico.append({
            "pergunta":    pergunta,
            "fluxo":       fluxo,
            "score":       resultado.get("melhor_score", 0.0),
            "filtro_time": filtro_ativo or filtro_time,
        })
        contadores[fluxo] = contadores.get(fluxo, 0) + 1

        if filtro_ativo:
            filtro_time = None


chatbot_interativo_suporte()


  ASSISTENTE DE SUPORTE TECNICO — MODULO DO ANALISTA

  Descreva o problema ou use um dos comandos abaixo:

  /time <nome>   filtra busca dentro de um time especifico
                 exemplo: /time FOLHA qual o procedimento para rubrica?
  /id <numero>   busca chamados similares a um ID existente
                 exemplo: /id 42013898
  /times         lista todos os times disponíveis
  historico      exibe interacoes desta sessao
  stats          estatisticas dos fluxos acionados
  sair           encerra a sessao
  ----------------------------------------------------------------------

  Analisando...

ASSISTENTE DE SUPORTE — MODULO DO ANALISTA
Fluxo         : Caso inedito
Score maximo  : 0.0%
----------------------------------------------------------------------

RESPOSTA:

CASO INEDITO

Nenhum chamado similar encontrado na base.
Recomenda-se abertura de chamado com descricao detalhada
e encaminhamento para triagem manual.

Tempo de resposta: 0.2s


KeyboardInterrupt: Interrupted by user

## 12 - Geração de Formulários para Avaliação Qualitativa (Monografia)
Esta seção extrai de forma balanceada uma amostra de 50 chamados (15 Simples, 20 Complexos e 15 Críticos) baseados em heurísticas, e automatiza a inferência de ambos os modelos de Chatbot (Interno e Público) para gerar formulários para a avaliação humana.

In [13]:
import pandas as pd
import numpy as np

COL_DUVIDA  = 'Descrição do chamado'
COL_SOLUCAO = 'Última ação de acompanhamento'
SEED = 42

# Usa a base unificada (que inclui MGI, Apple Store e Google Play)
df_work = df.copy()

# Tratamento de nulos
df_work[COL_DUVIDA]  = df_work[COL_DUVIDA].fillna('').astype(str).str.strip()
df_work[COL_SOLUCAO] = df_work[COL_SOLUCAO].fillna('').astype(str).str.strip()

# Máscaras de Utilidade
mask_sol_preenchida = (df_work[COL_SOLUCAO] != '') & (df_work[COL_SOLUCAO].str.lower() != 'nan')
mask_sol_detalhada  = df_work[COL_SOLUCAO].str.len() >= 30

# --- A: Casos Críticos / Vagos ---
kw_criticos = 'caiu|parou|urgente|servidor fora|não abre|trava|ruim|lixo'
mask_criticos_len = df_work[COL_DUVIDA].str.len() < 30
mask_criticos_kw  = df_work[COL_DUVIDA].str.contains(kw_criticos, case=False, na=False)
df_criticos_total = df_work[mask_criticos_len | mask_criticos_kw]
amostra_criticos  = df_criticos_total.sample(n=15, random_state=SEED)

# --- B: Incidentes Complexos ---
kw_complexos = 'erro|falha|sistema|integração|folha|integracao|carregar|bug|atualização'
mask_complexos_len = df_work[COL_DUVIDA].str.len() > 100
mask_complexos_kw  = df_work[COL_DUVIDA].str.contains(kw_complexos, case=False, na=False)
df_complexos_total = df_work[
    (mask_complexos_len) & (mask_complexos_kw) & (mask_sol_detalhada)
].drop(amostra_criticos.index, errors='ignore')
amostra_complexos = df_complexos_total.sample(n=20, random_state=SEED)

# --- C: Dúvidas Simples ---
kw_simples = 'senha|acesso|contracheque|desbloquear|entrar|login|cadastro'
mask_simples_len = df_work[COL_DUVIDA].str.len() < 100
mask_simples_kw  = df_work[COL_DUVIDA].str.contains(kw_simples, case=False, na=False)
indices_usados   = amostra_criticos.index.union(amostra_complexos.index)
df_simples_total = df_work[
    (mask_simples_len | mask_simples_kw) & (mask_sol_preenchida)
].drop(indices_usados, errors='ignore')
amostra_simples  = df_simples_total.sample(n=15, random_state=SEED)

# --- Distribuição em 5 formulários ---
formularios = {}
for i in range(5):
    slice_simples   = amostra_simples.iloc[i*3 : (i+1)*3]
    slice_complexos = amostra_complexos.iloc[i*4 : (i+1)*4]
    slice_criticos  = amostra_criticos.iloc[i*3 : (i+1)*3]

    form_atual = pd.concat([slice_simples, slice_complexos, slice_criticos])
    form_atual = form_atual.sample(frac=1, random_state=SEED+i).reset_index(drop=True)
    formularios[f'form_{i+1}'] = form_atual

print("✅ Amostragem de 50 chamados concluída e dividida em 5 formulários!")

KeyboardInterrupt: 

In [ ]:
from tqdm.auto import tqdm
import os

# Adaptadores para as funções RAG que já existem no seu notebook
def gerar_resposta_copiloto_interno(pergunta_usuario):
    # Usa a função RAG Suporte (Seção 11 do notebook)
    res = chatbot_rag_suporte(pergunta_usuario, top_k=10, verbose=False)
    return res['resposta']

def gerar_resposta_autoatendimento_publico(pergunta_usuario):
    # Usa a função RAG Público (Seção 9 do notebook)
    res = chatbot_rag(pergunta_usuario, top_k=10, verbose=False)
    return res['resposta']

# Cria a pasta para salvar os arquivos finais
os.makedirs('resultados_avaliacao', exist_ok=True)

for nome_form, df_form in formularios.items():
    print(f"\\n🚀 Processando {nome_form}...")
    respostas_internas = []
    respostas_publicas = []

    for index, row in tqdm(df_form.iterrows(), total=len(df_form), desc=nome_form):
        pergunta = row['Descrição do chamado']

        # Try/Except protege contra quebras (limite de contexto da GPU)
        try:
            resp_a = gerar_resposta_copiloto_interno(pergunta)
        except Exception as e:
            resp_a = f"[ERRO IA - INTERNO]: {e}"

        try:
            resp_b = gerar_resposta_autoatendimento_publico(pergunta)
        except Exception as e:
            resp_b = f"[ERRO IA - PÚBLICO]: {e}"

        respostas_internas.append(resp_a)
        respostas_publicas.append(resp_b)

    df_form['Resposta_Chatbot_Interno'] = respostas_internas
    df_form['Resposta_Chatbot_Publico'] = respostas_publicas

    # Exporta Excel Limpo com as Respostas
    nome_arquivo = f"resultados_avaliacao/{nome_form}_Tabela_IA.xlsx"
    df_form.to_excel(nome_arquivo, index=False)

print("\\n🎉 Inferência da IA concluída! Arquivos Excel salvos na pasta 'resultados_avaliacao'.")

In [ ]:
TEXTO_INTRODUCAO = """Seção 1 de 2
Validação Especializada de Inteligência Artificial: Gestão de Chamados da Folha (MGI)

Prezada equipe,
Estamos testando ferramentas de Inteligência Artificial para otimizar o atendimento dos chamados nos nossos sistemas estruturantes. Como a área de Folha de Pagamento tem alta demanda, sua expertise é fundamental para validarmos a qualidade técnica das respostas geradas pela IA.

Como vai funcionar? Você avaliará 10 cenários reais (anonimizados). Para cada um, a IA gerou duas respostas:
- Chatbot Interno: Um "copiloto" para a equipe técnica, que resume casos históricos parecidos para acelerar a resolução do chamado.
- Chatbot Público: Um "autoatendimento" focado em resolver a dúvida do usuário na origem, diminuindo a abertura de chamados simples.

O que avaliar? Para cada resposta, pedimos que você avalie 3 frentes (Compreensão, Utilidade/Eficácia e Clareza) atribuindo uma nota em uma escala de 1 a 5, onde 1 significa "Discordo Totalmente" e 5 significa "Concordo Totalmente".

Nota de Confidencialidade: Todos os dados pessoais dos chamados reais foram anonimizados.

Agradecemos imensamente o seu tempo!"""

def gerar_txts_formulario(dict_formularios):
    for nome_form, df_form in dict_formularios.items():
        caminho_arquivo = f"resultados_avaliacao/{nome_form}_texto_para_copiar.txt"

        with open(caminho_arquivo, 'w', encoding='utf-8') as f:
            f.write(TEXTO_INTRODUCAO + "\\n=========================================\\nSeção 2 de 2\\n")

            for i, row in df_form.iterrows():
                chamado = row.get('Descrição do chamado', '')
                resp_a  = row.get('Resposta_Chatbot_Interno', '')
                resp_b  = row.get('Resposta_Chatbot_Publico', '')

                bloco = f"""
-----------------------------------------------------------------------------
Cenário {i+1}/10
-----------------------------------------------------------------------------
CHAMADO ORIGINAL DO USUÁRIO:
{chamado}

RESPOSTA MODELO A (CHATBOT INTERNO)
Texto da IA:
{resp_a}

Avalie o Modelo A (Escala 1 a 5):
[ ] Compreensão e Relevância: O retorno demonstra que o chamado foi compreendido.
[ ] Utilidade Prática: O plano de ação sugerido aceleraria a resolução deste chamado.
[ ] Clareza e Objetividade: O texto vai direto ao ponto, sem redundâncias.

RESPOSTA MODELO B (CHATBOT PÚBLICO)
Texto da IA:
{resp_b}

Avalie o Modelo B (Escala 1 a 5):
[ ] Compreensão e Relevância: O retorno demonstra que o chamado foi compreendido.
[ ] Eficácia: A resposta resolve a dúvida de forma autônoma, evitando abertura de chamado.
[ ] Clareza e Objetividade: A linguagem é acessível e estruturada.
\\n"""
                f.write(bloco)
        print(f"✅ Arquivo de texto gerado: {caminho_arquivo}")

gerar_txts_formulario(formularios)
print("\\n✨ Processo finalizado! Acesse o ícone de 'Pastas' na barra lateral esquerda do Colab para baixar os arquivos Excel e TXT de dentro da pasta 'resultados_avaliacao'.")